# Weather Data

In [1]:
import cdsapi # important in order to use the CDS API
import pandas as pd
import numpy as np
import zipfile
import os
from glob import glob
import holidays

## CDS API Call
To make use of the CDSAPI and access the ERA5-Land dataset via the C3S data store, the following steps are mandatory.

### 1. Account & Credentials
Register for a free account on the Climate Data Store (CDS) website (cds.climate.copernicus.eu) or log in.
You can find your Personal Access Token in your profile (https://cds.climate.copernicus.eu/profile).

### 2. Create a .cdsapirc file
In the home directory ($HOME/.cdsapirc on Linux/Mac, or C:\Users\<Name>\.cdsapirc on Windows), create a file with the following content:

```
url: https://cds.climate.copernicus.eu/api
key: <YOUR-PERSONAL-ACCESS-TOKEN>
```
The cdsapi client automatically reads the token and URL from this file – therefore, in the code, cdsapi.Client() without any parameters is sufficient.

### 3. Install a Python package

```
bash
pip install "cdsapi>=0.7.7"
```
Older versions no longer work reliably with the new CDS system (since the 2024 migration), so please use the latest version.

### 4. Accept the licence terms
For each dataset (in this case, ERA5 Land hourly time-series data from 1950 to present), you must manually accept the Terms of Use on the website once. This can be done via the dataset’s download page, at the bottom of the form. Without this, the API request will fail, even if you have the correct key.


In [2]:

taxi_data_processed = pd.read_parquet('../data/processed/taxi_data_processed_big.parquet')

end_date = pd.to_datetime(
    taxi_data_processed["Trip End Timestamp"].max(),
    format="%m/%d/%Y %I:%M:%S %p"
).strftime("%Y-%m-%d")

print(end_date)

2026-05-13


In [3]:
# config dictionary for the CDS API call

CONFIG = {
    # select the variables we are interested in; find the names of the variables at https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview or make use of the web based dataset picker 
    # on https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=download 
    "variables": [
        "2m_temperature", 
        "total_precipitation",
        "snow_cover",
        "snow_depth",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind"
    ],
    "date": f"2024-01-01/{end_date}",
    "dir_name": "era5_data.zip",
}

In [4]:
# API call of the CDS API which can also be generated on https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=download

dataset = "reanalysis-era5-land-timeseries"

# use the values from the config dictionary 
request = {
    "variable": CONFIG["variables"],
    "location": {"longitude": -87.8, "latitude": 42}, # coordinates of Chicago
    "date": CONFIG["date"],
    "data_format": "csv"
}

client = cdsapi.Client()
client.retrieve(dataset, request).download(f"../data/{CONFIG['dir_name']}")

2026-07-26 16:43:28,785 INFO Request ID is acd0298c-1ca6-4692-b1e9-b3c8979a88e2
2026-07-26 16:43:28,873 INFO status has been updated to accepted
2026-07-26 16:43:43,627 INFO status has been updated to running
2026-07-26 16:44:02,779 INFO status has been updated to successful


b7f4e95a7e0fc59a4ec7f5759431993a.zip:   0%|          | 0.00/653k [00:00<?, ?B/s]

'../data/era5_data.zip'

## Load individual CSV files 

In [5]:

with zipfile.ZipFile(f"../data/{CONFIG['dir_name']}", "r") as zip_ref:
    zip_ref.extractall("era5_data")

print(os.listdir("era5_data"))

['reanalysis-era5-land-timeseries-sfc-2m-temperatureqrgx0jpe.csv', 'reanalysis-era5-land-timeseries-sfc-pressure-precipitationv5ou1g_1.csv', 'reanalysis-era5-land-timeseries-sfc-snowm6raz6cg.csv', 'reanalysis-era5-land-timeseries-sfc-wind4oo7938j.csv']


In [6]:

files = glob("era5_data/*.csv")

# dictionary for the dataframes
dfs = {}

for file in files:
    filename = os.path.basename(file)

    if "temperature" in filename:
        key = "temp"
    elif "precipitation" in filename:
        key = "precip"
    elif "snow" in filename:
        key = "snow"
    elif "wind" in filename:
        key = "wind"

    dfs[key] = pd.read_csv(file, encoding="latin1")

    print(f"\n--- {key} ---")
    print(dfs[key].head())


--- temp ---
            valid_time        t2m  latitude  longitude
0  2024-01-01 00:00:00  274.39615      42.0      -87.8
1  2024-01-01 01:00:00  274.24170      42.0      -87.8
2  2024-01-01 02:00:00  274.05762      42.0      -87.8
3  2024-01-01 03:00:00  273.90906      42.0      -87.8
4  2024-01-01 04:00:00  274.01672      42.0      -87.8

--- precip ---
            valid_time        tp  latitude  longitude
0  2024-01-01 00:00:00  0.000281      42.0      -87.8
1  2024-01-01 01:00:00  0.000150      42.0      -87.8
2  2024-01-01 02:00:00  0.000030      42.0      -87.8
3  2024-01-01 03:00:00  0.000014      42.0      -87.8
4  2024-01-01 04:00:00  0.000037      42.0      -87.8

--- snow ---
            valid_time     snowc       sde  latitude  longitude
0  2024-01-01 00:00:00  6.929688  0.007812      42.0      -87.8
1  2024-01-01 01:00:00  8.179688  0.008789      42.0      -87.8
2  2024-01-01 02:00:00  8.804688  0.008789      42.0      -87.8
3  2024-01-01 03:00:00  8.873047  0.008789    

## Merge individual Dataframes

In [7]:
df_merged = dfs["temp"].copy()

df_merged = df_merged.merge(
    dfs["precip"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)
df_merged = df_merged.merge(
    dfs["snow"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)
df_merged = df_merged.merge(
    dfs["wind"],
    on=["valid_time", "latitude", "longitude"],
    how="left",
    suffixes=("", "_snow")
)

df_merged

,valid_time,t2m,latitude,longitude,tp,snowc,sde,u10,v10
0,2024-01-01 00:00:00,274.39615,42.0,-87.8,0.000281,6.929688,7.812500e-03,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,42.0,-87.8,0.000150,8.179688,8.789062e-03,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,42.0,-87.8,0.000030,8.804688,8.789062e-03,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,42.0,-87.8,0.000014,8.873047,8.789062e-03,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,42.0,-87.8,0.000037,8.890625,8.789062e-03,2.084305,-6.889282
...,...,...,...,...,...,...,...,...,...
20731,2026-05-13 19:00:00,284.58185,42.0,-87.8,0.000005,0.000000,-7.345365e-24,-2.860031,-1.829824
20732,2026-05-13 20:00:00,284.27710,42.0,-87.8,0.000004,0.000000,-7.345365e-24,-3.003967,-1.621773
20733,2026-05-13 21:00:00,283.77032,42.0,-87.8,0.000003,0.000000,-7.345365e-24,-2.911026,-1.236721
20734,2026-05-13 22:00:00,283.36694,42.0,-87.8,0.000004,0.000000,-7.345365e-24,-2.610672,-0.892094


## Data Preprocessing

In [8]:
# check for missing values
print("--- Missing values ---")
print(df_merged.isna().sum())
print("")

# check for data types
print("--- Data types ---")
print(df_merged.dtypes)
print("")

# short statistical description of the data
print("--- Stat description ---")
print(df_merged.describe())

--- Missing values ---
valid_time    0
t2m           0
latitude      0
longitude     0
tp            0
snowc         0
sde           0
u10           0
v10           0
dtype: int64

--- Data types ---
valid_time        str
t2m           float64
latitude      float64
longitude     float64
tp            float64
snowc         float64
sde           float64
u10           float64
v10           float64
dtype: object

--- Stat description ---
                t2m      latitude     longitude            tp         snowc  \
count  20736.000000  2.073600e+04  2.073600e+04  2.073600e+04  20736.000000   
mean     283.154261  4.200000e+01 -8.780000e+01  1.108608e-04     10.299310   
std       10.778929  7.105599e-15  1.421120e-14  5.405291e-04     25.097608   
min      248.497120  4.200000e+01 -8.780000e+01 -3.736932e-08      0.000000   
25%      274.893000  4.200000e+01 -8.780000e+01  0.000000e+00      0.000000   
50%      283.363830  4.200000e+01 -8.780000e+01  0.000000e+00      0.000000   
75%      

### Clean up

In [9]:
# drop unnecessary columns as always the same coordinates for all data points
df_merged = df_merged.drop(['latitude', 'longitude'], axis=1)

# rename columns to improve readability and interpretabilty
df_merged = df_merged.rename(
    columns={
        'valid_time': 'time_step',
        't2m': '2m_temp',
        'tp': 'total_precip',
        'snowc': 'snow_cov',
        'sde': 'snow_depth'
    }
)
df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,7.812500e-03,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,8.789062e-03,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,8.789062e-03,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,8.789062e-03,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,8.789062e-03,2.084305,-6.889282
...,...,...,...,...,...,...,...
20731,2026-05-13 19:00:00,284.58185,0.000005,0.000000,-7.345365e-24,-2.860031,-1.829824
20732,2026-05-13 20:00:00,284.27710,0.000004,0.000000,-7.345365e-24,-3.003967,-1.621773
20733,2026-05-13 21:00:00,283.77032,0.000003,0.000000,-7.345365e-24,-2.911026,-1.236721
20734,2026-05-13 22:00:00,283.36694,0.000004,0.000000,-7.345365e-24,-2.610672,-0.892094


In [10]:
df_merged['time_step'] = pd.to_datetime(df_merged['time_step'])

df_merged['total_precip'] = df_merged['total_precip'].clip(lower=0)
df_merged['snow_depth'] = df_merged['snow_depth'].clip(lower=0)

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282
...,...,...,...,...,...,...,...
20731,2026-05-13 19:00:00,284.58185,0.000005,0.000000,0.000000,-2.860031,-1.829824
20732,2026-05-13 20:00:00,284.27710,0.000004,0.000000,0.000000,-3.003967,-1.621773
20733,2026-05-13 21:00:00,283.77032,0.000003,0.000000,0.000000,-2.911026,-1.236721
20734,2026-05-13 22:00:00,283.36694,0.000004,0.000000,0.000000,-2.610672,-0.892094


### Conversion of units

In [11]:
# convert temperature from Kelvin to Celsius via formula C = K - 273.15
df_merged['2m_temp_c'] = df_merged['2m_temp'] - 273.15

# convert from m to mm (common unit for precipitation)
df_merged['total_precip_mm'] = df_merged['total_precip'] * 1000

df_merged

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491
...,...,...,...,...,...,...,...,...,...
20731,2026-05-13 19:00:00,284.58185,0.000005,0.000000,0.000000,-2.860031,-1.829824,11.43185,0.004590
20732,2026-05-13 20:00:00,284.27710,0.000004,0.000000,0.000000,-3.003967,-1.621773,11.12710,0.004053
20733,2026-05-13 21:00:00,283.77032,0.000003,0.000000,0.000000,-2.911026,-1.236721,10.62032,0.003159
20734,2026-05-13 22:00:00,283.36694,0.000004,0.000000,0.000000,-2.610672,-0.892094,10.21694,0.003725


## Data Export
Export the data into its own parquet file in order to merge it with POI and taxi data later on.

In [12]:
# Export weather data table to use in other notebooks
df_merged.to_parquet("../data/processed/weather_data_processed.parquet")

# Tables for the Report

In [13]:
#| label: tbl-meteo-var
#| tbl-cap: "Selected Meteorological Variables"
from IPython.display import HTML
pd.set_option('display.max_colwidth', None)
table_weather = pd.DataFrame([
    {
        "Variable": "2m Temperature (K and °C)",
        "Description": "Air temperature at 2 meters above the surface. Expected to drive customer demand (e.g., fewer walk/bike substitutes in extreme cold) and EV battery performance, since range may drop at low temperatures.",
        "Datatype": "float64"
    },
    {
        "Variable": "Total Precipitation (m and mm)",
        "Description": "Accumulated liquid and solid precipitation reaching the surface. Expected to increase short-distance ride demand and reduce walking/cycling alternatives.",
        "Datatype": "float64"
    },
    {
        "Variable": "10m Wind Components u10 and v10",
        "Description": "Eastward and northward horizontal wind components at 10 meters above ground level. Combined into wind speed and direction (see Feature Engineering); relevant for extreme-weather demand spikes and outdoor-wait discomfort.",
        "Datatype": "float64"
    },
    {
        "Variable": "Snow Cover / Snow Depth",
        "Description": "Fraction of the grid cell covered by snow, and mean snow thickness on the ground (excluding canopy snow). Expected to directly affect road conditions, trip duration, and safety-related demand drops, and are key drivers of charging infrastructure planning in winter months.",
        "Datatype": "float64"
    }
])
# Relative column widths so the LaTeX/PDF table wraps long text instead of
# overflowing the page margin (pandoc turns <colgroup> widths into p{} columns).
_colgroup = '<colgroup><col style="width:22%"><col style="width:63%"><col style="width:15%"></colgroup>'
HTML(table_weather.to_html(escape=False, index=False).replace('<thead>', _colgroup + '<thead>', 1))


Variable,Description,Datatype
2m Temperature (K and °C),"Air temperature at 2 meters above the surface. Expected to drive customer demand (e.g., fewer walk/bike substitutes in extreme cold) and EV battery performance, since range may drop at low temperatures.",float64
Total Precipitation (m and mm),Accumulated liquid and solid precipitation reaching the surface. Expected to increase short-distance ride demand and reduce walking/cycling alternatives.,float64
10m Wind Components u10 and v10,Eastward and northward horizontal wind components at 10 meters above ground level. Combined into wind speed and direction (see Feature Engineering); relevant for extreme-weather demand spikes and outdoor-wait discomfort.,float64
Snow Cover / Snow Depth,"Fraction of the grid cell covered by snow, and mean snow thickness on the ground (excluding canopy snow). Expected to directly affect road conditions, trip duration, and safety-related demand drops, and are key drivers of charging infrastructure planning in winter months.",float64
